# Temporal Detection Evaluation

Evaluate cat and person tracks using temporal IoU, precision, recall and F1.

## Configure notebook

In [ ]:
from nb_utils import setup_notebook

PROJECT_ROOT = setup_notebook()

In [ ]:
import json
from collections import defaultdict
from collections.abc import Iterable
from pathlib import Path

import joblib
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from scipy.optimize import linear_sum_assignment
from optuna.trial import TrialState

import batch_run
from config import SETTINGS_PATH_OVERRIDE, settings
from nb_utils import (
    PERSON_LABEL_ID,
    VideoInfo,
    annotation_intervals,
    is_video_annotated,
    load_annotations,
)
from utils import get_video_paths

In [ ]:
# define constants
TIOU_THRESHOLDS = np.arange(0.5, 1.0, 0.1).round(1)
PRIMARY_TIOU_THRESHOLD = TIOU_THRESHOLDS[0]
N_TRIALS = 25

In [ ]:
# define paths
METADATA_DIR = PROJECT_ROOT / "object_clips" / "metadata"
LABELS_PATH = PROJECT_ROOT / "datasets" / "behaviour_dataset" / "behaviour_labels.json"
OPTIMISATION_CACHE_DIR = PROJECT_ROOT / ".cache" / "temporal_detection"

## Load target videos

In [ ]:
# all processed or annotated videos should be a subset of target videos
video_paths = get_video_paths(mock_inputs_long=True)
video_paths.sort(key=lambda p: p.stat().st_size)
video_info = {p.name: VideoInfo(p) for p in video_paths}

## Define functions

In [ ]:
# define functions
def temporal_iou(a: dict, b: dict) -> float:
    """calculate temporal intersection over union for inclusive frame intervals"""
    intersection = max(0, min(a["end"], b["end"]) - max(a["start"], b["start"]) + 1)
    union = (a["end"] - a["start"] + 1) + (b["end"] - b["start"] + 1) - intersection
    return intersection / union


def match_temporal_intervals(
    ground_truth: list[dict],
    predictions: list[dict],
    video_names: Iterable[str],
) -> dict:
    """match intervals one-to-one at each temporal IoU threshold"""
    matches_by_threshold = {t: [] for t in TIOU_THRESHOLDS}
    for video_name in video_names:
        for class_name in ("cat", "person"):
            # subset ground truth and predictions
            gt_sub = [
                r
                for r in ground_truth
                if r["video"] == video_name and r["class"] == class_name
            ]
            pred_sub = [
                r
                for r in predictions
                if r["video"] == video_name and r["class"] == class_name
            ]

            # calculate overlaps and identify matches
            if gt_sub and pred_sub:
                overlaps = np.array(
                    [[temporal_iou(g, p) for p in pred_sub] for g in gt_sub]
                )
                match_count = min(overlaps.shape)
                for threshold in TIOU_THRESHOLDS:
                    valid_matches = overlaps >= threshold
                    # an extra valid match must outweigh any possible total IoU difference
                    scores = np.where(valid_matches, match_count + 1 + overlaps, 0)
                    rows, columns = linear_sum_assignment(scores, maximize=True)
                    matches_by_threshold[threshold].extend(
                        (gt_sub[i], pred_sub[j])
                        for i, j in zip(rows, columns)
                        if valid_matches[i, j]
                    )

    return matches_by_threshold


def evaluate_temporal_predictions(
    ground_truth: list[dict],
    predictions: list[dict],
    video_names: Iterable[str],
) -> tuple[pd.DataFrame, dict]:
    """calculate class and pooled temporal detection metrics"""
    matches_by_threshold = match_temporal_intervals(
        ground_truth, predictions, video_names
    )
    # calculate counts for each threshold and class
    metrics = []
    for threshold, threshold_matches in matches_by_threshold.items():
        for class_name in ("cat", "person", "all"):
            gt_sub = [
                r
                for r in ground_truth
                if class_name == "all" or r["class"] == class_name
            ]
            pred_sub = [
                r
                for r in predictions
                if class_name == "all" or r["class"] == class_name
            ]
            true_positives = sum(
                class_name == "all" or g["class"] == class_name
                for g, _ in threshold_matches
            )
            metrics.append(
                {
                    "tIoU": threshold,
                    "class": class_name,
                    "ground-truth intervals": len(gt_sub),
                    "predicted tracks": len(pred_sub),
                    "TP": true_positives,
                }
            )

    # calculate metrics from counts
    metrics = pd.DataFrame(metrics).set_index(["tIoU", "class"])
    metrics["FP"] = metrics["predicted tracks"] - metrics["TP"]
    metrics["FN"] = metrics["ground-truth intervals"] - metrics["TP"]
    metrics["precision"] = metrics["TP"].div(metrics["predicted tracks"]).fillna(0)
    metrics["recall"] = metrics["TP"].div(metrics["ground-truth intervals"]).fillna(0)
    metrics["F-beta"] = (
        (3 * metrics["TP"])
        .div(3 * metrics["TP"] + metrics["FP"] + 2 * metrics["FN"])
        .fillna(0)
    )
    metrics["F1"] = (
        (2 * metrics["TP"])
        .div(2 * metrics["TP"] + metrics["FP"] + metrics["FN"])
        .fillna(0)
    )
    return metrics, matches_by_threshold


def build_ground_truth(
    annotations: list[dict],
    ann_video_info: dict[str, VideoInfo],
    stale_duration: float,
) -> list[dict]:
    """merge annotation windows into class-specific presence intervals"""
    # collect class-specific annotation windows
    windows = defaultdict(list)
    for video_name, video in ann_video_info.items():
        intervals = annotation_intervals(annotations, video.hashes, video_name)
        for cat_id, start, end, _ in intervals:
            class_name = "person" if cat_id == PERSON_LABEL_ID else "cat"
            identity = cat_id if class_name == "cat" else "person"
            windows[(video_name, class_name, identity)].append((start, end))

    # merge windows when the same object could remain in the track's stale state
    ground_truth = []
    for (name, class_name, _), intervals in windows.items():
        merged = []
        for start, end in sorted(intervals):
            gap = start - merged[-1][1] - 1 if merged else None
            if merged and gap < stale_duration * ann_video_info[name].fps:
                merged[-1] = (merged[-1][0], max(merged[-1][1], end))
            else:
                merged.append((start, end))
        ground_truth.extend(
            {"video": name, "class": class_name, "start": s, "end": e}
            for s, e in merged
        )

    return ground_truth


def load_tracks(metadata_dir: Path) -> list[dict]:
    """load exported tracks from the metadata directory"""
    return [joblib.load(p) for p in metadata_dir.glob("track-*.joblib")]


def load_prediction_intervals(
    tracks: list[dict], video_info: dict[str, VideoInfo]
) -> list[dict]:
    """convert exported tracks into source-frame intervals"""
    # build hash map for each video
    video_hash_indices = {
        n: {h: i for i, h in enumerate(v.hashes)} for n, v in video_info.items()
    }

    # calculate index based intervals
    predictions = []
    for track in tracks:
        name = Path(track["video_path"]).name
        if name in video_info:
            hash_indices = video_hash_indices[name]
            frame_indices = [
                hash_indices[h] for h in track["frame_hashes"] if h in hash_indices
            ]
            if frame_indices:
                predictions.append(
                    {
                        "video": name,
                        "class": track["object_name"],
                        "start": min(frame_indices),
                        "end": max(frame_indices),
                    }
                )

    return predictions

## Load ground truth

In [ ]:
# identify labelled target videos and report unlabelled videos
annotations = load_annotations(LABELS_PATH)
ann_video_info = {
    n: v for n, v in video_info.items() if is_video_annotated(v, annotations)
}
unlabelled_videos = set(video_info) - set(ann_video_info)
print(f"Unlabelled videos: {len(unlabelled_videos)}")
for video_name in sorted(unlabelled_videos):
    print(f"  {video_name}")

# merge annotation windows into presence intervals
ground_truth = build_ground_truth(annotations, ann_video_info, settings.TRACK_STALE_DUR)
print(f"Ground-truth intervals: {len(ground_truth)}")

## Run batch processing

In [ ]:
# apply evaluation-only settings to the child process
try:
    SETTINGS_PATH_OVERRIDE.write_text('LOG_LEVEL = "ERROR"\nEXCLUDED_OBJECTS = ""\n')
    return_code = batch_run.main(video_paths, clear_output=True)
finally:
    SETTINGS_PATH_OVERRIDE.unlink(missing_ok=True)

## Load predicted tracks

In [ ]:
# load exported tracks and report unprocessed target videos
tracks = load_tracks(METADATA_DIR)
proc_video_names = {
    n for t in tracks if (n := Path(t["video_path"]).name) in video_info
}
proc_video_info = {n: v for n, v in video_info.items() if n in proc_video_names}
unprocessed_videos = set(video_info) - set(proc_video_info)
print(f"Unprocessed videos: {len(unprocessed_videos)}")
for video_name in sorted(unprocessed_videos):
    print(f"  {video_name}")

# convert exported tracks into source-frame intervals
predictions = load_prediction_intervals(tracks, proc_video_info)

# report the number of predicted intervals
print(f"Predicted intervals: {len(predictions)}")

## Evaluate predictions

In [ ]:
# calculate metrics and matches for the configured thresholds
evaluation_predictions = [p for p in predictions if p["video"] in ann_video_info]
metrics, matches_by_threshold = evaluate_temporal_predictions(
    ground_truth, evaluation_predictions, ann_video_info
)

# use the display threshold for latency and per-video diagnostics
matches = matches_by_threshold[PRIMARY_TIOU_THRESHOLD]

# store detection latency for each matched track
detection_latencies = pd.DataFrame(
    [
        {
            "video": g["video"],
            "class": p["class"],
            "latency_seconds": (p["start"] - g["start"]) / video_info[p["video"]].fps,
        }
        for g, p in matches
    ],
)

# show performance per class aggregated across threshold
display(metrics.round(2))
metrics.groupby(
    ["class", "ground-truth intervals", "predicted tracks"]
).mean().reset_index(["ground-truth intervals", "predicted tracks"]).round(2)

## Plot per-video timelines

In [ ]:
# find videos where intervals and matches disagree
video_names_with_errors = []
for video_name in ann_video_info:
    truth_count = sum(r["video"] == video_name for r in ground_truth)
    track_count = sum(r["video"] == video_name for r in predictions)
    match_count = sum(g["video"] == video_name for g, _ in matches)
    if not truth_count == track_count == match_count:
        video_names_with_errors.append(video_name)

# compare ground-truth and predicted intervals for videos with errors
if video_names_with_errors:
    fig, axes = plt.subplots(
        len(video_names_with_errors),
        1,
        figsize=(14, 2.5 * len(video_names_with_errors)),
        squeeze=False,
    )

    matched_ground_truth = {id(g) for g, _ in matches}
    matched_predictions = {id(p) for _, p in matches}

    # iterate over videos and categories and intervals
    for v_ind, (axis, video_name) in enumerate(
        zip(axes[:, 0], video_names_with_errors)
    ):
        for lane, (class_name, intervals, colour, matched_ids) in enumerate(
            [
                ("cat", ground_truth, "tab:green", matched_ground_truth),
                ("cat", predictions, "tab:blue", matched_predictions),
                ("person", ground_truth, "tab:orange", matched_ground_truth),
                ("person", predictions, "tab:red", matched_predictions),
            ]
        ):
            lane_intervals = [
                i
                for i in intervals
                if i["video"] == video_name and i["class"] == class_name
            ]
            for c_ind, interval in enumerate(lane_intervals):

                # calculate offset for overlapping intervals
                overlapping_before = sum(
                    interval["start"] <= x["end"] and x["start"] <= interval["end"]
                    for x in lane_intervals[:c_ind]
                )
                overlaps_later = any(
                    interval["start"] <= x["end"] and x["start"] <= interval["end"]
                    for x in lane_intervals[c_ind + 1 :]
                )
                offset = 0
                if overlapping_before or overlaps_later:
                    offset = 0.15 * (-1) ** (overlapping_before + 1)

                # plot the intervals
                axis.plot(
                    [interval["start"], interval["end"]],
                    [lane + offset, lane + offset],
                    color=colour,
                    linewidth=3,
                    marker="o" if id(interval) in matched_ids else "^",
                    markersize=8,
                )

        # annotate the axis
        ticks = ["cat GT", "cat tracks", "person GT", "person tracks"]
        axis.set_yticks(range(4), ticks)
        axis.set_title(f"{v_ind}:   {video_name}")
        axis.set_xlabel("Source frame")
        axis.grid(axis="x", alpha=0.25)
    fig.tight_layout()

## Plot detection latency

In [ ]:
# compare matched detection latency between object classes
if not detection_latencies.empty:
    fig, ax = plt.subplots(figsize=(15, 4))
    sns.histplot(
        data=detection_latencies,
        x="latency_seconds",
        hue="class",
        stat="proportion",
        binwidth=0.2,
        common_norm=False,
        multiple="layer",
        element="step",
        fill=False,
        ax=ax,
    )
    ax.set_xlabel("Detection latency [s]")
    ax.set_ylabel("Count")
    ax.grid(axis="y")
    fig.tight_layout()

## Optimisation

In [ ]:
# define the full parameter search and baseline
WEIGHT_PARAMETER_NAMES = (
    "TRACK_MATCH_WEIGHT_IOU",
    "TRACK_MATCH_WEIGHT_CENTROID",
    "TRACK_MATCH_WEIGHT_CONFIDENCE",
    "TRACK_MATCH_WEIGHT_VISUAL",
    "TRACK_MATCH_WEIGHT_RECENCY",
    "TRACK_MATCH_WEIGHT_AGE",
    "TRACK_MATCH_WEIGHT_SIZE",
)
FLOAT_SEARCH_SPACE = {
    "CONF": (0.1, 0.35, 0.05),
    "TRACK_NEW_CONF_THRESHOLD": (0.25, 0.6, 0.05),
    "TRACK_NEW_DUR": (1, 5, 0.5),
    "TRACK_NEW_MIN_DETECTION_RATIO": (0.3, 0.8, 0.1),
    "TRACK_NEW_MIN_CONF_RATIO": (0.1, 0.5, 0.05),
    "TRACK_STALE_DUR": (3, 15, 1),
    "TRACK_MATCH_THRESHOLD": (0.4, 0.7, 0.05),
    "NMS_IOU_THRESHOLD": (0.2, 0.6, 0.1),
    "NMS_IOM_THRESHOLD": (0.6, 0.95, 0.05),
    **{n: (0.5, 5, 0.5) for n in WEIGHT_PARAMETER_NAMES},
}
BASELINE_PARAMS = {n: float(getattr(settings, n)) for n in FLOAT_SEARCH_SPACE}
BASELINE_PARAMS["VERIFY_DETECTIONS"] = bool(settings.VERIFY_DETECTIONS)
PENALTY_STEPS = {n: b[2] for n, b in FLOAT_SEARCH_SPACE.items()}
PENALTY_STEPS["VERIFY_DETECTIONS"] = 1

In [ ]:
# optimisation functions
def suggest_parameters(trial: optuna.trial.Trial) -> dict:
    """suggest a complete tracking configuration"""
    params = {
        n: trial.suggest_float(n, l, h, step=s)
        for n, (l, h, s) in FLOAT_SEARCH_SPACE.items()
    }
    params["VERIFY_DETECTIONS"] = trial.suggest_categorical(
        "VERIFY_DETECTIONS", [False, True]
    )
    return params


def parameter_penalty(params: dict) -> float:
    """calculate the per-parameter cost of deviating from baseline"""
    steps = [
        abs(float(params[n]) - float(BASELINE_PARAMS[n])) / s
        for n, s in PENALTY_STEPS.items()
    ]
    return 0.01 * np.mean(steps)


def run_optimisation() -> tuple[optuna.study.Study, pd.DataFrame]:
    """run or resume the persistent per-video study"""
    # create persistent study storage
    OPTIMISATION_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    storage_url = f"sqlite:///{(OPTIMISATION_CACHE_DIR / 'study.sqlite3').resolve()}"
    study = optuna.create_study(
        study_name="temporal_detection_f_beta_per_video",
        storage=storage_url,
        load_if_exists=True,
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
    )

    # retry failed and still-running trials
    retry_trials = study.get_trials(states=(TrialState.FAIL, TrialState.RUNNING))
    for t in retry_trials:
        study.enqueue_trial(t.params)

    def objective(trial: optuna.trial.Trial) -> float:
        """evaluate and optimise one tracking configuration"""
        parameters = suggest_parameters(trial)
        predictions = []

        def evaluate_progress(video_count: int) -> float:
            """evaluate the trial on the processed video prefix"""
            video_sub = [p.name for p in video_paths[:video_count]]
            gt_sub = [i for i in ground_truth if i["video"] in video_sub]
            trial_metrics, _ = evaluate_temporal_predictions(
                gt_sub, predictions, video_sub
            )
            score = trial_metrics["F-beta"].drop(index="all", level="class").mean()
            trial.report(score, step=video_count)
            return score

        # create a per-trial settings file for the child processes
        override = {"LOG_LEVEL": "ERROR", "EXCLUDED_OBJECTS": "", **parameters}
        override_text = "\n".join(f"{n} = {json.dumps(v)}" for n, v in override.items())
        SETTINGS_PATH_OVERRIDE.write_text(f"{override_text}\n")

        # process and evaluate each video
        try:
            for index, video_path in enumerate(video_paths):
                message = f"Trial {trial.number}, video {index + 1}/{len(video_paths)}: {video_path.name}"
                print(message)
                return_code = batch_run.main([video_path], clear_output=True)
                if return_code == 0:
                    tracks = load_tracks(METADATA_DIR)
                    video_predictions = load_prediction_intervals(
                        tracks, {video_path.name: video_info[video_path.name]}
                    )
                    predictions.extend(video_predictions)
                else:
                    message = f"Skipping {video_path.name}: retries exhausted (code {return_code}); no detections"
                    print(message)

                score = evaluate_progress(index + 1)
                if trial.should_prune():
                    raise optuna.TrialPruned()

            trial.set_user_attr("pre-penalty macro-averaged F-beta", score)
            return score - parameter_penalty(parameters)
        finally:
            SETTINGS_PATH_OVERRIDE.unlink(missing_ok=True)

    # continue to the configured count of complete or pruned trials
    completed_trials = sum(
        t.state in (TrialState.COMPLETE, TrialState.PRUNED) for t in study.trials
    )
    remaining_trials = max(N_TRIALS - completed_trials, 0) + len(retry_trials)
    study.optimize(objective, n_trials=remaining_trials)

    # display the best fully evaluated candidate
    trial_results = study.trials_dataframe(
        attrs=("number", "state", "value", "params", "user_attrs"), multi_index=False
    ).rename(
        columns={
            "number": "trial",
            "value": "objective score",
            "user_attrs_pre-penalty macro-averaged F-beta": "pre-penalty macro-averaged F-beta",
        }
    )
    trial_results = trial_results.sort_values("objective score", ascending=False)
    return study, trial_results

In [ ]:
# run optimisation and display results
study, optimisation_trial_results = run_optimisation()
display(optimisation_trial_results.head())
display(pd.Series(study.best_params))